# 9장 — 훈련 일지 판독법: 손실 곡선 읽기 🩺

4대 증상(스파이크·평원·계단·벌어짐)을 전부 재현하고, 처방까지 확인합니다.

**이 노트북의 목표**
1. 증상 도감 — 각 무늬를 앞 장들의 무대에서 직접 만들어 본다
2. 그래디언트 클리핑 — 방향 유지·크기 절단의 산수와 스파이크 보험 효과
3. 진단 규칙 — 모양이 용의자를, 시점이 다이얼을 지목한다

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch9')

## 1. 증상 ⚡ 스파이크 — 그리고 클리핑 보험

이차 골짜기 SGD에 드문 이상치 배치(노이즈 ×30)를 심습니다 — 클리핑 유무를 비교하세요.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def spike_run(seed, outs, clip):
    rng = np.random.default_rng(seed)
    w, losses = 6.0, []
    for t in range(160):
        noise = 0.3 * rng.standard_normal()
        if t in outs: noise *= 30          # 이상치 배치
        g = 2 * w + noise
        if clip: g = np.sign(g) * min(abs(g), 3.0)   # 그래디언트 클리핑 |g| ≤ 3
        w -= 0.15 * g
        losses.append(w * w)
    return np.array(losses)

no = spike_run(92, {55, 120}, False)
yes = spike_run(92, {55, 120}, True)
base = np.median(no[30:55])

plt.figure(figsize=(7.5, 3))
plt.semilogy(np.maximum(no, 1e-5), color='tab:red', label='클리핑 없음')
plt.semilogy(np.maximum(yes, 1e-5), color='tab:green', label='클리핑 |g| ≤ 3')
plt.xlabel('스텝'); plt.legend(); plt.grid(alpha=0.3); plt.title('⚡ 스파이크와 보험')
plt.tight_layout(); plt.show()

print(f'평시 손실(중앙값) {base:.4f} | 무클립 스파이크 max {no[50:].max():.2f} ({no[50:].max()/base:.0f}배)')
print(f'클리핑 후 max {yes[50:].max():.3f} — 스파이크가 {no[50:].max()/yes[50:].max():.1f}분의 1')

assert no[50:].max() > 100 * base, '이상치 배치 하나가 손실을 수백 배 치솟게 한다'
assert no[50:].max() > 4 * yes[50:].max(), '클리핑이 스파이크 크기를 크게 줄인다'
assert yes[159] < 0.01 and no[159] < 0.01, '둘 다 결국 회복 — 스파이크(경고)는 발산(사고)과 등급이 다르다'
print('\n✔ 확인: 처방 = 클리핑(상시 보험). 초반에만 몰리면 워밍업(6장) 쪽')

## 2. 증상 🏜️ 평원 — 개입으로 범인 구분

높은 손실에서 평평 — 보폭 과소인지 지형 평원인지, lr을 올려 개입해 봅니다.

In [ ]:
def plateau_run(seed, lr):
    rng = np.random.default_rng(seed)
    w, losses = 8.0, []
    for t in range(160):
        w -= lr * (2 * w + 0.1 * rng.standard_normal())
        losses.append(w * w)
    return np.array(losses)

flat = plateau_run(94, 0.0006)   # 굶주린 보폭
fixed = plateau_run(94, 0.06)    # 개입: lr ×100
print(f'lr 0.0006: {flat[0]:.1f} → {flat[-1]:.1f} (160스텝에 거의 제자리)')
print(f'lr 0.06  : {fixed[0]:.1f} → {fixed[-1]:.5f} (즉시 반응 — 범인은 보폭 과소)')

assert flat[-1] > 0.5 * flat[0], '보폭 과소: 절반도 못 내려감 — 평평해 보인다'
assert fixed[-1] < 0.01, '개입에 즉시 반응 = 기울기는 살아 있었다'
print('\n✔ 확인: 개입에 반응하면 보폭 과소(2장), 무반응이면 지형 평원(1장·10장) — 진단의 절반은 개입 실험')

## 3. 증상 🪜 계단 + ↔️ 벌어짐

계단은 스텝 감쇠 일정표(6장)의 지문 — 병이 아닙니다. 벌어짐은 7장의 그 훈련을 재현합니다.

In [ ]:
# 계단 — lr 0.3 → 0.09 → 0.027 (스텝 55, 105에서 감쇠)
rng = np.random.default_rng(96)
w, stair, lrs = 8.0, [], []
for t in range(160):
    lr = 0.3 if t < 55 else (0.09 if t < 105 else 0.027)
    w -= lr * (2 * w + 1.2 * rng.standard_normal())
    stair.append(w * w); lrs.append(lr)
stair = np.array(stair)
s1, s2, s3 = np.median(stair[15:55]), np.median(stair[65:105]), np.median(stair[115:160])

# 벌어짐 — 7장 차수 9 GD (시드 21/22, 체비쇼프)
def make_data(n, seed):
    r = np.random.default_rng(seed)
    x = np.clip(np.linspace(-1, 1, n) + r.uniform(-0.04, 0.04, n), -1, 1)
    return x, np.sin(2.5 * x) + 0.25 * r.standard_normal(n)
def cheb(x):
    T = [np.ones_like(x), x]
    for k in range(2, 10): T.append(2 * x * T[-1] - T[-2])
    return np.stack(T, axis=1)
xt, yt = make_data(10, 21); xv, yv = make_data(30, 22)
Ft, Fv = cheb(xt), cheb(xv)
wv, trc, vac = np.zeros(10), [], []
for t in range(3000):
    r = Ft @ wv - yt
    wv -= 0.05 * 2 * Ft.T @ r / 10
    trc.append(np.mean(r ** 2)); vac.append(np.mean((Fv @ wv - yv) ** 2))
trc, vac = np.array(trc), np.array(vac); k = int(vac.argmin())

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].semilogy(stair, color='indigo'); ax2 = axes[0].twinx(); ax2.plot(lrs, color='tab:cyan', lw=1)
axes[0].set_title('🪜 계단 = lr 일정표의 지문 (하늘색 = lr)')
axes[1].plot(trc, color='indigo', alpha=0.4, label='훈련'); axes[1].plot(vac, color='tab:red', label='검증')
axes[1].axvline(k, color='green', ls=':'); axes[1].set_ylim(0, 0.6); axes[1].legend()
axes[1].set_title('↔️ 벌어짐 = 과적합 시작 (7장)')
plt.tight_layout(); plt.show()

print(f'계단 구간 중앙값: {s1:.3f} → {s2:.3f} → {s3:.4f} (lr 감쇠 시점과 일치)')
print(f'벌어짐: 검증 최저 {vac[k]:.4f}@{k} → 마지막 {vac[-1]:.4f} ({vac[-1]/vac[k]:.1f}배)')

assert s1 > 2 * s2 and s2 > 2 * s3, '계단: lr이 떨어질 때마다 손실 바닥도 한 단 — 서성임 반경 ∝ 보폭(6장)'
assert k < 500 and vac[-1] > 2 * vac[k], '벌어짐: 검증 최저 후 되오름 — 과적합(7장), 처방은 8장'
print('\n✔ 확인: 계단은 정상 무늬(처방 불필요), 벌어짐은 진행 중인 병(조기 종료·weight decay)')

## 4. 클리핑의 산수 — 방향 유지, 크기 절단

In [ ]:
g = np.array([3.0, 4.0])
max_norm = 1.0
norm = np.linalg.norm(g)          # 1부의 그 노름 — 3-4-5 삼각형!
scale = max_norm / norm
g_clip = g * scale
print(f'‖g‖ = {norm} > {max_norm} → 배율 {scale} → 클리핑 후 {g_clip} (노름 {np.linalg.norm(g_clip):.1f})')

assert norm == 5.0 and scale == 0.2, 'StepQuiz 값'
assert np.allclose(g_clip, [0.6, 0.8]) and np.isclose(np.linalg.norm(g_clip), 1.0)
assert np.isclose(g_clip[1] / g_clip[0], g[1] / g[0]), '방향(성분 비율)은 그대로 — 정보는 살리고 사고만 막는다'
print('\n✔ 확인: g × (한계/‖g‖) — 설정 파일의 max_grad_norm: 1.0이 이 산수입니다')

## 5. 관찰 정리 + 🤖 AI 연결

- **4대 증상 진단표**: ⚡스파이크 = 보폭×기울기의 순간 사고(처방 클리핑·lr↓, 초반 한정이면
  워밍업) / 🏜️평원 = 보폭 과소 or 지형 평원(개입으로 구분) / 🪜계단 = 스텝 감쇠의 정상
  지문 / ↔️벌어짐 = 과적합 시작(조기 종료·weight decay·데이터).
- **진단 규칙**: 모양이 용의자를, 시점이 다이얼을 지목한다. 판독 순서 = 모양 → 시점 →
  용의자 → 개입(한 번에 다이얼 하나 — 변인 통제).
- **그래디언트 클리핑**: 노름 초과 시 방향 유지·크기 절단 — `max_grad_norm: 1.0`.
  스파이크 등급: 회복 = 경고 / NaN = 사고(체크포인트에서 lr 낮춰 재시작).
- 회수: 6부 9장 조련 성적표의 시간축 버전(eval_loss 곡선) — 실무 도구는 wandb·TensorBoard.

**다음 챕터**: 10장 — 지형의 함정들. 수십억 차원의 산에는 왜 갇힐 골짜기보다 안장점이
많은가, 그런데도 SGD는 왜 통하는가 (7부 4장 고차원 직관 회수).